### Batch Ingestion

In [0]:
taxi_path = "/Volumes/unified_lakehouse/raw_files/landing/yellow_tripdata_2024-01.parquet"
users_path = "/Volumes/unified_lakehouse/raw_files/landing/users.avro"

df_taxi = spark.read.parquet(taxi_path)
df_taxi.write.format("delta").mode("overwrite").saveAsTable("unified_lakehouse.bronze.taxi")

df_users = spark.read.format("avro").load(users_path)
df_users.write.format("delta").mode("overwrite").saveAsTable("unified_lakehouse.bronze.users")

### Streaming Ingestion

In [0]:
%pip install faker

I added `awaitTermination()` function because since free trial does not support endless streaming, I had to use `availableNow` which only generated a very few number of rows. By waiting for **240** seconds, it generated **2732** rows which is moderate and satisfiable number of rows in respect with its execution time.

In [0]:
from faker import Faker
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

drivers_schema = StructType([
    StructField("name", StringType(), True),
    StructField("car_number", StringType(), True),
    StructField("experience", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
])

def generate_drivers(batches):
    fake = Faker()
    for pdf in batches:
        n = len(pdf)
        yield pd.DataFrame({
            "name": [fake.name() for _ in range(n)],
            "car_number": [fake.license_plate() for _ in range(n)],
            "experience": [fake.random_int(min=0, max=50) for _ in range(n)],
            "rating": [round(fake.pyfloat(min_value=0.0, max_value=5.0), 1) for _ in range(n)],
            "timestamp": pd.to_datetime(pdf["timestamp"]),
        })

checkpoint_path = "/Volumes/unified_lakehouse/raw_files/landing/checkpoints/drivers"

drivers_stream = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .load()
        .select("timestamp")
        .mapInPandas(generate_drivers, schema=drivers_schema)
)

query = (
    drivers_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .table("unified_lakehouse.bronze.drivers")
)

query.awaitTermination(240)
query.stop()

### Schema Evolution

*5700 rows in total, 2968 with phone number, 2732 with null values for phone column*

In [0]:
from faker import Faker
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

drivers_schema = StructType([
    StructField("name", StringType(), True),
    StructField("car_number", StringType(), True),
    StructField("experience", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("phone", StringType(), True),
])

def generate_drivers(batches):
    fake = Faker()
    for pdf in batches:
        n = len(pdf)
        yield pd.DataFrame({
            "name": [fake.name() for _ in range(n)],
            "car_number": [fake.license_plate() for _ in range(n)],
            "experience": [fake.random_int(min=0, max=50) for _ in range(n)],
            "rating": [round(fake.pyfloat(min_value=0.0, max_value=5.0), 1) for _ in range(n)],
            "timestamp": pd.to_datetime(pdf["timestamp"]),
            "phone": [fake.phone_number() for _ in range(n)],
        })

checkpoint_path = "/Volumes/unified_lakehouse/raw_files/landing/checkpoints/drivers"

drivers_stream = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .load()
        .select("timestamp")
        .mapInPandas(generate_drivers, schema=drivers_schema)
)

query = (
    drivers_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("unified_lakehouse.bronze.drivers")
)

query.awaitTermination(240)
query.stop()

### Versioning and restoring

In [0]:
%sql
DESCRIBE HISTORY unified_lakehouse.bronze.drivers;

In [0]:
%sql
SELECT * FROM unified_lakehouse.bronze.drivers VERSION AS OF 2 LIMIT 5;
RESTORE unified_lakehouse.bronze.drivers VERSION AS OF 2;